# NoiseBridge — Kaggle Notebook (corrected)

**Model:** NoiseBridge (PWNIC + multi-task aux, no GRL)
**Training:** ONE unified run on mixed noise, evaluated on all 4 test sets.

**Setup:**
1. Kaggle → Settings → Accelerator → **GPU T4 x2** (or P100)
2. Add your `hindimix-data` dataset (Add Data → Your Datasets)
3. Change `ENCODER` and `SEED` in the config cell below
4. Run cells 1 → 6 to start an interactive session, then **Save Version** (Save & Run All) at cell 7 — it will run in the background for ~3h while you can still watch logs in the interactive session.

**Output:** `/kaggle/working/noisebridge_results.zip`

**Per-account assignments (3 encoders × 3 seeds = 9 sessions):**
| Account | ENCODER | SEED |
|---|---|---|
| 1 | `xlm-roberta-base` | 42 |
| 2 | `xlm-roberta-base` | 43 |
| 3 | `xlm-roberta-base` | 44 |
| 4 | `bert-base-multilingual-cased` | 42 |
| 5 | `bert-base-multilingual-cased` | 43 |
| 6 | `bert-base-multilingual-cased` | 44 |
| 7 | `ai4bharat/indic-bert` | 42 |
| 8 | `ai4bharat/indic-bert` | 43 |
| 9 | `ai4bharat/indic-bert` | 44 |

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.40.0", "jellyfish", "sentencepiece", "accelerate"], check=True)

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

import jellyfish
print('jellyfish OK:', jellyfish.jaro_winkler_similarity('test', 'test'))

In [ ]:
# ─── EDIT PER ACCOUNT ───
ENCODER = 'xlm-roberta-base'     # 'xlm-roberta-base' | 'bert-base-multilingual-cased' | 'ai4bharat/indic-bert'
SEED    = 42                      # 42 | 43 | 44

# ─── Hyperparameters (should not need to change) ───
EPOCHS      = 5
BATCH_SIZE  = 16                  # XLM-R/mBERT/MuRIL comfortable at 16 on T4/P100 with fp16
MAX_LEN     = 128
LR          = 3e-5
ALPHA       = 0.15                # PWNIC weight
GAMMA       = 0.2                 # aux noise-prediction weight
FP16        = True
NUM_WORKERS = 2

# ─── Data paths ───
import os

DATASET_NAME = 'hindimix-data'
DATA_DIR = f'/kaggle/input/{DATASET_NAME}'
if not os.path.exists(DATA_DIR):
    for d in os.listdir('/kaggle/input/'):
        if 'hindimix' in d.lower() or 'hinglish' in d.lower() or 'noisy' in d.lower():
            DATA_DIR = f'/kaggle/input/{d}'
            break

RESULTS_DIR = '/kaggle/working/results'
CKPT_DIR    = '/kaggle/working/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

print('Data dir:', DATA_DIR)
print('Files:')
for f in sorted(os.listdir(DATA_DIR)):
    print(f'  {f}')

## NoiseBridge Model (PWNIC + multi-task aux, NO GRL)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel, AutoModel
import jellyfish


# ─── Phonetic utilities ───
def phonetic_similarity(text_a, text_b):
    words_a = str(text_a).lower().split()
    words_b = str(text_b).lower().split()
    if not words_a or not words_b:
        return 1.0
    n = max(len(words_a), len(words_b))
    scores = []
    for i in range(min(len(words_a), len(words_b))):
        try:
            scores.append(jellyfish.jaro_winkler_similarity(words_a[i], words_b[i]))
        except Exception:
            scores.append(1.0)
    scores += [0.0] * (n - len(scores))
    return sum(scores) / n


def phonetic_weights(clean_texts, noisy_texts):
    weights = [1.0 - phonetic_similarity(c, n) for c, n in zip(clean_texts, noisy_texts)]
    return torch.tensor(weights, dtype=torch.float32)


# ─── PWNIC Loss (phonetic-weighted contrastive, InfoNCE style) ───
class PWNICLoss(nn.Module):
    def __init__(self, temperature=0.15):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_clean, z_noisy, phi):
        N = z_clean.size(0)
        device = z_clean.device
        if N < 2:
            return torch.zeros((), device=device, requires_grad=True)

        z_clean = F.normalize(z_clean, dim=1)
        z_noisy = F.normalize(z_noisy, dim=1)
        z_all   = torch.cat([z_clean, z_noisy], dim=0)

        sim = torch.mm(z_all, z_all.T) / self.temperature
        sim.masked_fill_(torch.eye(2*N, dtype=torch.bool, device=device), float('-inf'))

        pos_idx = torch.cat([
            torch.arange(N, 2*N, device=device),
            torch.arange(0, N,   device=device),
        ])
        pos_sim   = sim[torch.arange(2*N, device=device), pos_idx]
        log_denom = torch.logsumexp(sim, dim=1)
        loss_each = -(pos_sim - log_denom)

        phi      = phi.to(device)
        phi_both = torch.cat([phi, phi], dim=0)
        wsum     = phi_both.sum().clamp(min=1e-8)
        return (phi_both * loss_each).sum() / wsum


# ─── Learned token gate ───
class NoiseAwareAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.gate = nn.Linear(hidden_size, 1)

    def forward(self, h):
        return h * torch.sigmoid(self.gate(h))


# ─── NoiseBridge ───
class NoiseBridge(nn.Module):
    NOISE_LEVEL_MAP = {'clean': 0, 'low': 1, 'medium': 2, 'high': 3}

    def __init__(self, encoder_name='xlm-roberta-base', num_labels=2,
                 num_noise_levels=4, dropout=0.1, temperature=0.15,
                 alpha=0.15, gamma=0.2, enable_aux=True):
        super().__init__()
        self.alpha      = alpha
        self.gamma      = gamma
        self.enable_aux = enable_aux

        if 'byt5' in encoder_name.lower() or 't5' in encoder_name.lower():
            self.encoder     = T5EncoderModel.from_pretrained(encoder_name)
            self.hidden_size = self.encoder.config.d_model
        else:
            self.encoder     = AutoModel.from_pretrained(encoder_name)
            self.hidden_size = self.encoder.config.hidden_size

        self.noise_attention = NoiseAwareAttention(self.hidden_size)

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

        self.proj_head = nn.Sequential(
            nn.Linear(self.hidden_size, 128), nn.ReLU(), nn.Linear(128, 128),
        )

        self.aux_noise_predictor = nn.Sequential(
            nn.Linear(self.hidden_size, 64), nn.ReLU(),
            nn.Linear(64, num_noise_levels),
        )

        self.pwnic = PWNICLoss(temperature=temperature)

    def encode(self, input_ids, attention_mask):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = self.noise_attention(out.last_hidden_state)
        mask   = attention_mask.unsqueeze(-1).float()
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def predict(self, input_ids, attention_mask):
        z = self.encode(input_ids, attention_mask)
        return self.classifier(self.dropout(z))

    def forward(self, clean_input_ids, clean_attention_mask,
                noisy_input_ids, noisy_attention_mask,
                labels, ce_mask_clean, ce_mask_noisy, has_pair,
                phi, noise_level_labels, class_weights=None):
        device = clean_input_ids.device
        B      = labels.size(0)

        z_clean      = self.encode(clean_input_ids, clean_attention_mask)
        logits_clean = self.classifier(self.dropout(z_clean))

        ce_c_per   = F.cross_entropy(logits_clean, labels, weight=class_weights, reduction='none')
        ce_per_row = ce_c_per * ce_mask_clean.float()

        loss_pwnic = torch.zeros((), device=device)
        loss_aux   = torch.zeros((), device=device)

        need_noisy = bool((has_pair | ce_mask_noisy).any().item())
        if need_noisy:
            z_noisy      = self.encode(noisy_input_ids, noisy_attention_mask)
            logits_noisy = self.classifier(self.dropout(z_noisy))

            ce_n_per   = F.cross_entropy(logits_noisy, labels, weight=class_weights, reduction='none')
            ce_per_row = ce_per_row + ce_n_per * ce_mask_noisy.float()

            paired_idx = has_pair.nonzero(as_tuple=True)[0]

            if paired_idx.numel() >= 2:
                p_clean    = self.proj_head(z_clean[paired_idx])
                p_noisy    = self.proj_head(z_noisy[paired_idx])
                phi_paired = phi[paired_idx]
                loss_pwnic = self.pwnic(p_clean, p_noisy, phi_paired)

            if self.enable_aux and paired_idx.numel() > 0:
                aux_logits  = self.aux_noise_predictor(z_noisy[paired_idx])
                aux_targets = noise_level_labels[paired_idx]
                loss_aux    = F.cross_entropy(aux_logits, aux_targets)

        loss_ce    = ce_per_row.sum() / max(B, 1)
        total_loss = loss_ce + self.alpha * loss_pwnic + self.gamma * loss_aux

        return {
            'loss':       total_loss,
            'loss_ce':    loss_ce,
            'loss_pwnic': loss_pwnic,
            'loss_aux':   loss_aux,
            'logits':     logits_clean,
        }

print('NoiseBridge model defined.')

## Dataset + Training Utilities

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NOISE_LEVEL_MAP = {'clean': 0, 'low': 1, 'medium': 2, 'high': 3}


# ─── Reproducibility ───
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = (torch.initial_seed() + worker_id) % (2**32)
    random.seed(worker_seed); np.random.seed(worker_seed)


# ─── Per-row anchor dataset (CE count matches baseline exactly) ───
class NoiseBridgeDataset(Dataset):
    """
    Each item = one row a baseline trainer would see. Per-row CE mask
    selects which embedding (clean or noisy) gets CE. Paired rows
    additionally get PWNIC + aux.

    Pairing: clean.text <-> noisy.text_original
    (in the provided data, clean rows have text_original=NaN while
     noisy rows store the clean text in text_original).
    """

    def __init__(self, train_df, noise_level, tokenizer, max_len=128):
        if 'text_original' not in train_df.columns:
            raise ValueError("train.csv must contain 'text_original'")

        train_df = train_df.dropna(subset=['text', 'label']).copy()

        clean_df = train_df[train_df['noise_level'] == 'clean']
        if noise_level == 'clean':
            noisy_df = train_df.iloc[0:0]
        elif noise_level == 'all':
            noisy_df = train_df[train_df['noise_level'].isin(['low', 'medium', 'high'])]
        else:
            noisy_df = train_df[train_df['noise_level'] == noise_level]

        anchor_df = clean_df if noise_level == 'clean' else pd.concat([clean_df, noisy_df], ignore_index=True)

        clean_text_set = set(clean_df['text'].dropna().astype(str))
        noisy_lookup = {}
        for _, r in noisy_df.iterrows():
            orig_text = r.get('text_original')
            if pd.isna(orig_text): continue
            orig_text = str(orig_text)
            if orig_text not in noisy_lookup:
                noisy_lookup[orig_text] = (str(r['text']), str(r['noise_level']).lower())

        clean_texts, noisy_texts = [], []
        labels, noise_ids = [], []
        ce_mask_clean, ce_mask_noisy, has_pair = [], [], []

        for _, row in anchor_df.iterrows():
            anchor_text  = str(row['text'])
            anchor_label = int(row['label'])
            anchor_level = str(row['noise_level']).lower()

            if anchor_level == 'clean':
                partner = noisy_lookup.get(anchor_text)
                if partner is not None:
                    p_text, p_level = partner
                    clean_texts.append(anchor_text); noisy_texts.append(p_text)
                    has_pair.append(True); noise_ids.append(NOISE_LEVEL_MAP[p_level])
                else:
                    clean_texts.append(anchor_text); noisy_texts.append(anchor_text)
                    has_pair.append(False); noise_ids.append(0)
                ce_mask_clean.append(True); ce_mask_noisy.append(False)
            else:
                orig = row.get('text_original')
                if pd.notna(orig) and str(orig) in clean_text_set:
                    clean_texts.append(str(orig)); noisy_texts.append(anchor_text)
                    has_pair.append(True); noise_ids.append(NOISE_LEVEL_MAP[anchor_level])
                else:
                    clean_texts.append(anchor_text); noisy_texts.append(anchor_text)
                    has_pair.append(False); noise_ids.append(NOISE_LEVEL_MAP[anchor_level])
                ce_mask_clean.append(False); ce_mask_noisy.append(True)

            labels.append(anchor_label)

        print(f'  Tokenizing {len(clean_texts):,} examples (max_len={max_len})...')
        self.clean_enc = tokenizer(clean_texts, max_length=max_len,
                                   padding='max_length', truncation=True, return_tensors='pt')
        self.noisy_enc = tokenizer(noisy_texts, max_length=max_len,
                                   padding='max_length', truncation=True, return_tensors='pt')

        print(f'  Computing phonetic weights...')
        self.phi = phonetic_weights(clean_texts, noisy_texts)

        self.labels        = torch.tensor(labels,        dtype=torch.long)
        self.noise_ids     = torch.tensor(noise_ids,     dtype=torch.long)
        self.ce_mask_clean = torch.tensor(ce_mask_clean, dtype=torch.bool)
        self.ce_mask_noisy = torch.tensor(ce_mask_noisy, dtype=torch.bool)
        self.has_pair      = torch.tensor(has_pair,      dtype=torch.bool)

        n_pair = int(self.has_pair.sum().item())
        n_solo = len(self.labels) - n_pair

        unique_noise = set()
        if n_pair > 0:
            unique_noise = set(self.noise_ids[self.has_pair].tolist())
        self.has_diverse_noise = len(unique_noise) >= 2

        print(f'  Dataset: {n_pair:,} paired + {n_solo:,} solo = {len(self):,} items')
        print(f'  CE supervisions / epoch: {len(self):,}  (matches baseline)')
        print(f'  Distinct noise levels in pairs: {sorted(unique_noise)} → aux: {"ENABLED" if self.has_diverse_noise else "disabled"}')

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'clean_input_ids':       self.clean_enc['input_ids'][idx],
            'clean_attention_mask':  self.clean_enc['attention_mask'][idx],
            'noisy_input_ids':       self.noisy_enc['input_ids'][idx],
            'noisy_attention_mask':  self.noisy_enc['attention_mask'][idx],
            'labels':                self.labels[idx],
            'noise_level_labels':    self.noise_ids[idx],
            'ce_mask_clean':         self.ce_mask_clean[idx],
            'ce_mask_noisy':         self.ce_mask_noisy[idx],
            'has_pair':              self.has_pair[idx],
            'phi':                   self.phi[idx],
        }


class EvalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.enc = tokenizer(texts, max_length=max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      self.enc['input_ids'][idx],
            'attention_mask': self.enc['attention_mask'][idx],
            'labels':         self.labels[idx],
        }


def collate_train(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

def collate_eval(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


# ─── Data loader ───
def load_data(noise_level):
    train_df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['text', 'label'])
    val_df   = pd.read_csv(f'{DATA_DIR}/val.csv').dropna(subset=['text', 'label'])

    conditions = ['clean', 'low', 'medium', 'high'] if noise_level == 'all' else [noise_level]
    test_dfs = {}
    for cond in conditions:
        path = f'{DATA_DIR}/test_clean.csv' if cond == 'clean' else f'{DATA_DIR}/test_noisy_{cond}.csv'
        if os.path.exists(path):
            test_dfs[cond] = pd.read_csv(path).dropna(subset=['text', 'label'])
    return train_df, val_df, test_dfs


# ─── Train / eval loops ───
def train_epoch(model, loader, optimizer, scheduler, class_weights, scaler=None):
    model.train()
    totals = {'loss': 0.0, 'ce': 0.0, 'pwnic': 0.0, 'aux': 0.0}
    for batch in tqdm(loader, desc='  train', leave=False):
        optimizer.zero_grad()
        kwargs = {k: v.to(DEVICE) for k, v in batch.items()}
        kwargs['class_weights'] = class_weights

        if scaler is not None:
            with autocast(device_type='cuda'):
                out = model(**kwargs)
            scaler.scale(out['loss']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(**kwargs)
            out['loss'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        totals['loss']  += out['loss'].item()
        totals['ce']    += out['loss_ce'].item()
        totals['pwnic'] += out['loss_pwnic'].item()
        totals['aux']   += out['loss_aux'].item()

    n = max(len(loader), 1)
    return {k: v / n for k, v in totals.items()}


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  eval', leave=False):
            logits = model.predict(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE),
            )
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            targets.extend(batch['labels'].tolist())
    return f1_score(targets, preds, average='macro'), preds, targets

print('Dataset and utilities defined.')

## Train NoiseBridge — one unified run, eval on all 4 test sets
**~2.5–3 hours on T4 with fp16.** Click **Save Version → Save & Run All** here so the run continues in the background; logs also stream live in this interactive session.

In [ ]:
set_seed(SEED)

print(f'{"="*72}')
print(f'NoiseBridge | encoder={ENCODER} | seed={SEED}')
print(f'alpha={ALPHA} (PWNIC) | gamma={GAMMA} (aux) | fp16={FP16}')
print(f'Mode: UNIFIED — one model on mixed noise, eval on all 4 test sets')
print(f'{"="*72}')

train_df, val_df, test_dfs = load_data('all')
tokenizer = AutoTokenizer.from_pretrained(ENCODER)

train_ds = NoiseBridgeDataset(train_df, 'all', tokenizer, max_len=MAX_LEN)
val_ds   = EvalDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer, max_len=MAX_LEN)
test_loaders = {}
for cond, tdf in test_dfs.items():
    tds = EvalDataset(tdf['text'].tolist(), tdf['label'].tolist(), tokenizer, max_len=MAX_LEN)
    test_loaders[cond] = DataLoader(tds, batch_size=BATCH_SIZE*2, collate_fn=collate_eval,
                                    num_workers=NUM_WORKERS, worker_init_fn=seed_worker)

print(f'  Val: {len(val_ds):,} | Test sets: {list(test_dfs.keys())}')

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=train_ds.labels.numpy())
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f'  Class weights: [{cw[0]:.3f}, {cw[1]:.3f}]')

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_train, num_workers=NUM_WORKERS,
                          pin_memory=True, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, collate_fn=collate_eval,
                        num_workers=NUM_WORKERS, worker_init_fn=seed_worker)

enable_aux = train_ds.has_diverse_noise

model = NoiseBridge(encoder_name=ENCODER, alpha=ALPHA, gamma=GAMMA, enable_aux=enable_aux).to(DEVICE)

if hasattr(model.encoder, 'gradient_checkpointing_enable'):
    model.encoder.gradient_checkpointing_enable()
    print('  Gradient checkpointing enabled.')

print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)

scaler = GradScaler('cuda') if FP16 else None

short_name  = ENCODER.split('/')[-1].replace('-', '_')
run_name    = f'noisebridge_{short_name}_all_s{SEED}'
ckpt_path   = f'{CKPT_DIR}/{run_name}.pt'
best_val_f1 = 0.0

for epoch in range(EPOCHS):
    stats = train_epoch(model, train_loader, optimizer, scheduler, class_weights, scaler)
    val_f1, _, _ = evaluate(model, val_loader)
    print(f'  Epoch {epoch+1}/{EPOCHS} | loss:{stats["loss"]:.4f} ce:{stats["ce"]:.3f} '
          f'pwnic:{stats["pwnic"]:.3f} aux:{stats["aux"]:.3f} | val F1:{val_f1:.4f}')
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), ckpt_path)
        print(f'    -> best saved (val F1: {best_val_f1:.4f})')

print(f'\n{"="*72}')
print('EVALUATING best checkpoint on all test sets')
print(f'{"="*72}')

model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
all_results = []
for cond, tloader in test_loaders.items():
    test_f1, test_preds, test_targets = evaluate(model, tloader)
    print(f'\n[{cond}] Test F1 (macro): {test_f1:.4f}')
    print(classification_report(test_targets, test_preds, target_names=['Non-hate', 'Hate']))

    result = {
        'model': run_name, 'encoder': ENCODER, 'train_noise': 'all', 'test_noise': cond,
        'seed': SEED, 'test_f1_macro': round(test_f1, 4),
        'best_val_f1': round(best_val_f1, 4),
        'alpha': ALPHA, 'gamma': GAMMA, 'enable_aux': enable_aux,
        'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN,
    }
    with open(f'{RESULTS_DIR}/{run_name}_eval_{cond}.json', 'w') as f:
        json.dump(result, f, indent=2)
    all_results.append(result)

print('\nDone.')

## Summary + Download

In [ ]:
print('\n--- NoiseBridge Results ---')
print(f'{"train":<8} {"test":<10} {"seed":>5} {"val F1":>9} {"test F1":>9}')
for r in all_results:
    print(f'  {r["train_noise"]:<6} {r["test_noise"]:<10} {r["seed"]:>5} '
          f'{r["best_val_f1"]:>9.4f} {r["test_f1_macro"]:>9.4f}')

import shutil
shutil.make_archive('/kaggle/working/noisebridge_results', 'zip', RESULTS_DIR)
print('\nZipped -> /kaggle/working/noisebridge_results.zip')
print('Download from Output tab ->')